# Exploración del FASTA

**Responsable:** Thomas

**Objetivo:** conocer el FASTA antes de cargarlo y registrar sus inconsistencias, sin modificar los datos. La exploración del CSV está en `00_exploracion_csv.ipynb` (Juan Arias).

Modelo de referencia: `../MODELO.md`

## 0. Configuración

Rutas a los datos. Los archivos deben estar en `data/`.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO

FASTA = Path("../data/PanTEon_Database_v1.6.1.fasta")
CSV = Path("../data/PanTEon_Database_metadata_v1.6.1.csv")
RESULTADOS = Path("../resultados")

print("FASTA existe:", FASTA.exists(), "|", round(FASTA.stat().st_size / 1e6), "MB")

## 1. Lectura del FASTA (una sola pasada)

Se recorre el archivo registro por registro, sin cargar las secuencias en memoria. De cada registro se separa el encabezado (`ID#CLASE/ORDEN/SUPERFAMILIA @Especie`) y se calculan los atributos: **longitud**, **contenido GC** (`(G+C)/(A+C+G+T)`) y **% de N**. Tarda unos minutos.

No se modifica nada: solo se mide.

In [ ]:
# No guarda las secuencias en memoria: solo lo que se mide de cada una.
filas = []
caracteres = Counter()

for rec in SeqIO.parse(FASTA, "fasta"):
    header = rec.description          # encabezado completo, sin ">"
    seq = str(rec.seq)

    # Formato esperado: ID#CLASE/ORDEN/SUPERFAMILIA @Especie
    izq, especie = header.split("@", 1) if "@" in header else (header, "")
    pdb, clasif = izq.split("#", 1) if "#" in izq else (izq, "")
    niveles = [x for x in clasif.strip().split("/") if x] if clasif.strip() else []

    caracteres.update(seq)
    up = seq.upper()
    a, c, g, t, n = (up.count(b) for b in "ACGTN")
    validas = a + c + g + t

    filas.append({
        "pdb_id": pdb.strip(),
        "header_ok": "#" in header and "@" in header,
        "clasificacion": clasif.strip(),
        "n_niveles": len(niveles),
        "te_class": niveles[0] if len(niveles) > 0 else None,
        "te_order": niveles[1] if len(niveles) > 1 else None,
        "te_superfamily": niveles[2] if len(niveles) > 2 else None,
        "especie_raw": especie,
        "especie": " ".join(especie.split()),   # sin espacios sobrantes
        "length": len(seq),
        "gc": (g + c) / validas if validas else None,
        "n_pct": n / len(seq) if seq else None,
        "otros_chars": len(seq) - validas - n,  # IUPAC u otros
        "minusculas": seq != up,
    })

df = pd.DataFrame(filas)
print("Registros leídos:", len(df))
df.head()

## 2. Encabezados e identificadores

¿Todos los encabezados tienen `#` y `@`? ¿Los PDB tienen el formato esperado y son únicos?

In [ ]:
print("Encabezados sin '#' o sin '@':", (~df.header_ok).sum())
print("PDB con formato distinto a PDB + 8 dígitos:",
      (~df.pdb_id.str.fullmatch(r"PDB\d{8}")).sum())
print("PDB duplicados:", df.pdb_id.duplicated().sum())
display(df[~df.header_ok | df.pdb_id.duplicated(keep=False)].head(20))

## 3. Clasificación del TE

¿Todas tienen 3 niveles (Clase/Orden/Superfamilia)? ¿Qué valores hay en cada nivel? ¿Hay valores desconocidos?

In [ ]:
print("Niveles de clasificación por registro:")
display(df.n_niveles.value_counts().sort_index())
display(df[df.n_niveles != 3][["pdb_id", "clasificacion"]].head(20))

print("Clases:")
display(df.te_class.value_counts(dropna=False))
print("Órdenes:")
display(df.te_order.value_counts(dropna=False))
print("Superfamilias distintas:", df.te_superfamily.nunique())
display(df.te_superfamily.value_counts(dropna=False).head(30))

# Valores sospechosos (desconocidos o vacíos) en cualquier nivel
sospechosos = df.clasificacion.str.contains(r"(?i)unknown|unclass|^na$|/na/|none|\?", regex=True)
print("Clasificaciones sospechosas:", sospechosos.sum())
display(df[sospechosos].clasificacion.value_counts().head(20))

## 4. Especies

Especies distintas, espacios sobrantes y una vista rápida del cruce con el CSV. La lista se guarda en `resultados/especies_fasta.csv` para Juan Arias.

In [ ]:
print("Especies distintas en el FASTA:", df.especie.nunique())
print("Registros con espacios sobrantes en la especie:", (df.especie_raw != df.especie).sum())

conteo_especies = df.especie.value_counts().rename_axis("especie").reset_index(name="n_tes")
conteo_especies.to_csv(RESULTADOS / "especies_fasta.csv", index=False)   # para Juan Arias
print("Guardado resultados/especies_fasta.csv")
display(conteo_especies.head(10))
display(conteo_especies.tail(10))

# Vista rápida del cruce con el CSV (el análisis completo lo hace Juan Arias)
csv_especies = set(pd.read_csv(CSV, encoding="utf-8-sig")["Species"].astype(str).str.split().str.join(" ").str.lower())
fasta_especies = set(df.especie.str.lower())
print("Especies del FASTA que no están en el CSV:", len(fasta_especies - csv_especies))
print("Especies del CSV sin TEs en el FASTA:", len(csv_especies - fasta_especies))
print(sorted(fasta_especies - csv_especies)[:20])

## 5. Longitudes

Define cuánto conviene el diccionario disperso para los k-mers y si hay secuencias demasiado cortas.

In [ ]:
display(df.length.describe(percentiles=[.05, .25, .5, .75, .95]).round(1))
print("Secuencias vacías:", (df.length == 0).sum())
print("Secuencias con longitud < 6 (sin 6-mers):", (df.length < 6).sum())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.length[df.length > 0], bins=100, log=True)
ax.set_xscale("log")
ax.set_xlabel("Longitud (nt, escala log)")
ax.set_ylabel("Número de secuencias (escala log)")
ax.set_title("Distribución de longitudes de los TEs")
plt.show()

## 6. Caracteres

¿Solo A, C, G, T y N? ¿Hay minúsculas u otros códigos IUPAC (R, Y, K...)?

In [ ]:
print("Caracteres presentes y cuántas veces aparecen:")
display(pd.Series(caracteres).sort_values(ascending=False))
print("Secuencias con minúsculas:", df.minusculas.sum())
print("Secuencias con N:", (df.n_pct > 0).sum())
print("Secuencias con otros caracteres (no ACGTN):", (df.otros_chars > 0).sum())
display(df[df.n_pct > 0].n_pct.describe().round(4))

## 7. Atributos calculados (GC y % de N)

In [ ]:
display(df[["length", "gc", "n_pct"]].describe().round(4))
display(df.groupby("te_class")[["length", "gc"]].mean().round(3))

## 8. Resumen de inconsistencias

Se guarda en `resultados/inconsistencias_fasta.csv`. Debajo, anotar la **decisión** para cada problema: va directo al artículo.

In [ ]:
resumen = pd.DataFrame([
    ("Encabezados sin '#' o '@'", (~df.header_ok).sum()),
    ("PDB con formato inválido", (~df.pdb_id.str.fullmatch(r"PDB\d{8}")).sum()),
    ("PDB duplicados", df.pdb_id.duplicated().sum()),
    ("Clasificación con niveles distintos de 3", (df.n_niveles != 3).sum()),
    ("Clasificaciones sospechosas (unknown, NA...)", sospechosos.sum()),
    ("Registros con espacios sobrantes en la especie", (df.especie_raw != df.especie).sum()),
    ("Especies del FASTA sin metadatos en el CSV", len(fasta_especies - csv_especies)),
    ("Secuencias vacías", (df.length == 0).sum()),
    ("Secuencias de longitud < 6", (df.length < 6).sum()),
    ("Secuencias con minúsculas", df.minusculas.sum()),
    ("Secuencias con N", (df.n_pct > 0).sum()),
    ("Secuencias con caracteres distintos de ACGTN", (df.otros_chars > 0).sum()),
], columns=["Problema", "Casos"])
resumen.to_csv(RESULTADOS / "inconsistencias_fasta.csv", index=False)
resumen

## 9. Decisiones

| Problema | Casos | Decisión | Justificación |
|---|---|---|---|
| | | | |